## Association & Frequent-Pattern Mining
Reproduces basket-level association analysis (Apriori & FP-Growth), using `valid_purchase_lines` persisted from `preprocessing.ipynb`.

In [1]:
import pandas as pd

valid_purchase_lines = pd.read_csv("../data/processed/valid_purchase_lines.csv")
print(valid_purchase_lines.shape)
valid_purchase_lines[['InvoiceNo','StockCode','Description','Quantity']].head()

(349203, 15)


,InvoiceNo,StockCode,Description,Quantity
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6
1,536365,71053,WHITE METAL LANTERN,6
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6


In [2]:
# Stock codes that are NOT purely numeric-with-optional-letter-suffix (the normal product pattern)
# Normal: 85123, 85123A, 21730 etc. Administrative: POST, DOT, M, BANK CHARGES, AMAZONFEE, etc.
import re

def looks_like_product_code(code):
    return bool(re.match(r'^\d{4,6}[A-Za-z]?$', str(code)))

valid_purchase_lines['LooksLikeProduct'] = valid_purchase_lines['StockCode'].apply(looks_like_product_code)

non_product_codes = valid_purchase_lines[~valid_purchase_lines['LooksLikeProduct']]
print("Rows with non-standard stock codes:", non_product_codes.shape[0])
print(non_product_codes.groupby('StockCode')['Description'].first().sort_index())

Rows with non-standard stock codes: 560
StockCode
15056BL            EDWARDIAN PARASOL BLACK
BANK CHARGES                  Bank Charges
C2                                CARRIAGE
DOT                         DOTCOM POSTAGE
M                                   Manual
PADS            PADS TO MATCH ALL CUSHIONS
POST                               POSTAGE
Name: Description, dtype: object


In [3]:
# ---- Documented administrative code exclusion ----
# Confirmed non-product codes by manual inspection of descriptions:
ADMIN_CODES = ['BANK CHARGES', 'C2', 'DOT', 'M', 'POST']

print("Rows excluded as administrative:", valid_purchase_lines['StockCode'].isin(ADMIN_CODES).sum())

basket_source = valid_purchase_lines[~valid_purchase_lines['StockCode'].isin(ADMIN_CODES)].copy()
print("basket_source shape:", basket_source.shape)

Rows excluded as administrative: 328
basket_source shape: (348875, 16)


## Administrative Code Exclusion

Manually inspected 560 rows with non-standard StockCode patterns. Confirmed 5 codes as genuinely administrative (fees/postage, not purchasable products): `BANK CHARGES`, `C2` (carriage), `DOT` (dotcom postage), `M` (manual adjustment), `POST` (postage). Two initially-flagged codes (`15056BL`, `PADS`) were confirmed as real products via their descriptions and retained.

In [4]:
# ---- Step 1: distinct products per invoice (set, not list — dedupes automatically) ----
invoice_baskets_series = basket_source.groupby('InvoiceNo')['StockCode'].apply(lambda x: sorted(set(x)))

print("Number of invoices (baskets):", len(invoice_baskets_series))
print("Basket size distribution:")
print(invoice_baskets_series.apply(len).describe())

# quick look
invoice_baskets_series.head()

Number of invoices (baskets): 16579
Basket size distribution:
count    16579.000000
mean        20.756740
std         23.935496
min          1.000000
25%          6.000000
50%         15.000000
75%         27.000000
max        540.000000
Name: StockCode, dtype: float64


InvoiceNo
536365    [21730, 22752, 71053, 84029E, 84029G, 84406B, ...
536366                                       [22632, 22633]
536367    [21754, 21755, 21777, 22310, 22622, 22623, 227...
536368                         [22912, 22913, 22914, 22960]
536369                                              [21756]
Name: StockCode, dtype: object

In [5]:
from mlxtend.preprocessing import TransactionEncoder

baskets_list = invoice_baskets_series.tolist()

te = TransactionEncoder()
te_array = te.fit(baskets_list).transform(baskets_list)
basket_matrix = pd.DataFrame(te_array, columns=te.columns_, index=invoice_baskets_series.index)

print(basket_matrix.shape)
print("Memory usage (MB):", basket_matrix.memory_usage(deep=True).sum() / 1e6)

(16579, 3640)
Memory usage (MB): 60.480192


In [6]:
import os
os.makedirs("../data/processed", exist_ok=True)

# Save baskets as list-of-lists (compact) rather than the full one-hot matrix (huge/sparse)
invoice_baskets_df = invoice_baskets_series.reset_index()
invoice_baskets_df.columns = ['InvoiceNo', 'StockCodes']
invoice_baskets_df.to_csv("../data/processed/invoice_baskets.csv", index=False)
print("Saved invoice_baskets.csv")

Saved invoice_baskets.csv


In [8]:
from mlxtend.frequent_patterns import apriori
import time

support_candidates = [0.05, 0.03, 0.02, 0.015, 0.01]

sensitivity_results = []
for min_sup in support_candidates:
    try:
        start = time.time()
        freq_itemsets = apriori(basket_matrix, min_support=min_sup, use_colnames=True, low_memory=True)
        elapsed = time.time() - start
        sensitivity_results.append({
            'min_support': min_sup,
            'n_itemsets': len(freq_itemsets),
            'max_itemset_size': freq_itemsets['itemsets'].apply(len).max() if len(freq_itemsets) > 0 else 0,
            'time_seconds': round(elapsed, 2)
        })
        print(f"min_support={min_sup}: {len(freq_itemsets)} itemsets, {elapsed:.2f}s")
    except MemoryError:
        print(f"min_support={min_sup}: MemoryError — too low, stopping here")
        break

sensitivity_df = pd.DataFrame(sensitivity_results)
print(sensitivity_df)

min_support=0.05: 22 itemsets, 0.34s
min_support=0.03: 94 itemsets, 0.65s
min_support=0.02: 248 itemsets, 1.20s
min_support=0.015: 472 itemsets, 1.45s
min_support=0.01: 1051 itemsets, 2.12s
   min_support  n_itemsets  max_itemset_size  time_seconds
0        0.050          22                 1          0.34
1        0.030          94                 2          0.65
2        0.020         248                 3          1.20
3        0.015         472                 3          1.45
4        0.010        1051                 4          2.12
